In [40]:
import pandas as pd
import geopandas as gpd
import numpy as np
import folium
from folium.features import GeoJsonTooltip
import branca
from folium import FeatureGroup
from folium.plugins import Fullscreen, MeasureControl

In [41]:
tree_clusters = gpd.read_file("data/tree_cluster_polygons.geojson")
poi_clusters = gpd.read_file("data/poi_cluster_polygons.geojson")
crime = gpd.read_file("data/crimes_with_station_lists.geojson")
poi = gpd.read_file("data/poi_updated.geojson")
tree_clusters = tree_clusters.to_crs("EPSG:4326")
poi_clusters = poi_clusters.to_crs("EPSG:4326")
crime = crime.to_crs("EPSG:4326")
poi = poi.to_crs("EPSG:4326")

c:\Users\Jason\AppData\Local\Programs\Python\Python313\Lib\site-packages\geopandas\io\file.py:576: UserWarning: Could not parse column 'stations_in_radius_dist_m' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)


In [42]:
def _map_center_from_gdf(gdf: gpd.GeoDataFrame):
    """Return (lat, lon) center based on total bounds."""
    minx, miny, maxx, maxy = gdf.total_bounds
    return [(miny + maxy) / 2, (minx + maxx) / 2]

def _add_points_layer(m: folium.Map, gdf: gpd.GeoDataFrame, *, color: str, name: str,
                      radius: int = 3, opacity: float = 0.8, tooltip_cols=None):
    """Add a point layer as CircleMarkers."""
    fg = FeatureGroup(name=name, show=True)
    cols = [c for c in gdf.columns if c != "geometry"]
    if tooltip_cols is None:
        tooltip_cols = cols[:6]
    for _, row in gdf.iterrows():
        geom = row.geometry
        if geom is None or geom.is_empty:
            continue
        # handle Point and MultiPoint
        if geom.geom_type == "Point":
            pts = [geom]
        elif geom.geom_type == "MultiPoint":
            pts = list(geom.geoms)
        else:
            continue

        props = {c: row.get(c, None) for c in tooltip_cols if c in row.index}
        tooltip_html = "<br>".join([f"<b>{k}</b>: {props[k]}" for k in props if props[k] is not None])

        for pt in pts:
            folium.CircleMarker(
                location=[pt.y, pt.x],
                radius=radius,
                color=color,
                weight=1,
                fill=True,
                fill_color=color,
                fill_opacity=opacity,
                tooltip=folium.Tooltip(tooltip_html, sticky=True) if tooltip_html else None,
            ).add_to(fg)
    fg.add_to(m)
    return fg

In [43]:
print("Rows:", len(poi_clusters))
print("Missing geometry:", poi_clusters.geometry.isna().sum())
print("Empty geometry:", poi_clusters.geometry.is_empty.sum())
print("Valid geometry:", poi_clusters.is_valid.sum(), "out of", len(poi_clusters))

Rows: 158
Missing geometry: 0
Empty geometry: 0
Valid geometry: 158 out of 158


In [44]:
# ---------- Tree color scale ----------
if "tree_count" not in tree_clusters.columns:
    raise KeyError("Expected a 'tree_count' column in tree_clusters.")

tc_min = float(tree_clusters["tree_count"].min())
tc_max = float(tree_clusters["tree_count"].max())

if tc_min == tc_max:
    tc_max = tc_min + 1

greens = branca.colormap.linear.Greens_09.scale(tc_min, tc_max)
greens.caption = "Tree count per cluster"

def tree_style(feature):
    v = feature["properties"].get("tree_count", None)
    try:
        v = float(v)
    except (TypeError, ValueError):
        v = tc_min
    c = greens(v)
    return {
        "fillColor": c,
        "color": "green",
        "weight": 1,
        "fillOpacity": 0.75,
    }

# ---------- POI color scale ----------
if "poi_count" not in poi_clusters.columns:
    raise KeyError("Expected a 'poi_count' column in poi_clusters.")

pc_min = float(poi_clusters["poi_count"].min())
pc_max = float(poi_clusters["poi_count"].max())

if pc_min == pc_max:
    pc_max = pc_min + 1

purples = branca.colormap.linear.Purples_09.scale(pc_min, pc_max)
purples.caption = "POI count per cluster"

def poi_style(feature):
    v = feature["properties"].get("poi_count", None)
    try:
        v = float(v)
    except (TypeError, ValueError):
        v = pc_min
    c = purples(v)
    return {
        "fillColor": c,
        "color": "purple",
        "weight": 1,
        "fillOpacity": 0.75,
    }

# ---------- Tree-only map ----------
m_trees = folium.Map(
    location=_map_center_from_gdf(tree_clusters),
    zoom_start=12,
    tiles="CartoDB positron"
)

folium.GeoJson(
    tree_clusters,
    name="Tree clusters",
    style_function=tree_style,
    highlight_function=lambda f: {"weight": 3, "fillOpacity": 0.75},
    tooltip=GeoJsonTooltip(
        fields=[c for c in ["cluster_id", "tree_count"] if c in tree_clusters.columns]
    ),
).add_to(m_trees)

greens.add_to(m_trees)
Fullscreen().add_to(m_trees)
MeasureControl(position="bottomleft").add_to(m_trees)
folium.LayerControl(collapsed=False).add_to(m_trees)

In [45]:
# Center using the union of bounds across all datasets
all_bounds = np.vstack([
    tree_clusters.total_bounds,
    poi_clusters.total_bounds,
    crime.total_bounds,
    poi.total_bounds
])

minx, miny = all_bounds[:, 0].min(), all_bounds[:, 1].min()
maxx, maxy = all_bounds[:, 2].max(), all_bounds[:, 3].max()
center_all = [(miny + maxy) / 2, (minx + maxx) / 2]

m_all = folium.Map(location=center_all, zoom_start=12, tiles="CartoDB dark_matter")

# POI polygon clusters (same idea as tree clusters)
folium.GeoJson(
    poi_clusters,
    name="POI clusters",
    style_function=poi_style,
    highlight_function=lambda f: {"weight": 3, "fillOpacity": 0.85},
    tooltip=GeoJsonTooltip(
        fields=[c for c in ["poi_centroid_id", "poi_count", "dominant_poi_cluster"] if c in poi_clusters.columns]
    ),
).add_to(m_all)

# POI points (blue)
_add_points_layer(
    m_all,
    poi,
    color="blue",
    name="POIs",
    radius=3,
    opacity=0.85,
    tooltip_cols=[c for c in ["name", "amenity", "shop", "leisure", "tourism"] if c in poi.columns]
)

# Tree polygons
folium.GeoJson(
    tree_clusters,
    name="Tree clusters",
    style_function=tree_style,
    highlight_function=lambda f: {"weight": 3, "fillOpacity": 0.85},
    tooltip=GeoJsonTooltip(
        fields=[c for c in ["cluster_id", "tree_count"] if c in tree_clusters.columns]
    ),
).add_to(m_all)

# Crime points (red)
_add_points_layer(
    m_all,
    crime,
    color="red",
    name="Crime",
    radius=3,
    opacity=0.85,
    tooltip_cols=[c for c in ["OFFENSE_DESCRIPTION", "HIGHEST_NIBRS_DESCRIPTION"] if c in crime.columns]
)

greens.add_to(m_all)
purples.add_to(m_all)
Fullscreen().add_to(m_all)
MeasureControl(position="bottomleft").add_to(m_all)
folium.LayerControl(collapsed=False).add_to(m_all)

m_all.save("geo_map.html")